In [6]:
import pygame
import random
import sys
import math
import csv
import os
import traceback

# Initialize Pygame
pygame.init()

# Screen dimensions
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Square Survival Game")

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (255, 0, 0)
GREEN = (0, 255, 0)

# Clock
clock = pygame.time.Clock()
FPS = 60

# Load sound
collision_sound = pygame.mixer.Sound("collision.wav")

# Folder for recordings
RECORD_FOLDER = "recordings"
os.makedirs(RECORD_FOLDER, exist_ok=True)

# ---------------------- Utility functions ---------------------- #
def record_inputs_to_a_file(data, filename):
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["frame", "x", "y"])
        for frame, (x, y) in enumerate(data):
            writer.writerow([frame, x, y])

def load_recording(filename):
    positions = []
    with open(filename, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            positions.append((int(row["x"]), int(row["y"])))
    return positions
    
def calculate_position_at_border(x, y, dx, dy):
    if dx != 0:
        ticks_x = -x / dx if dx < 0 else (WIDTH - x) / dx
    else:
        ticks_x = float("inf")
    if dy != 0:
        ticks_y = -y / dy if dy < 0 else (HEIGHT - y) / dy
    else:
        ticks_y = float("inf")
    ticks = min(ticks_x, ticks_y)
    x += dx * ticks
    y += dy * ticks
    return x, y

# ---------------------- Classes ---------------------- #
class Player:
    def __init__(self, x, y, size, speed=5, full_health=5):
        self.x = x
        self.y = y
        self.size = size
        self.speed = speed
        self.full_health = full_health
        self.health = full_health

    def move(self, keys, boundary_x, boundary_y, boundary_size):
        if keys[pygame.K_w] or keys[pygame.K_UP]:
            self.y -= self.speed
        if keys[pygame.K_s] or keys[pygame.K_DOWN]:
            self.y += self.speed
        if keys[pygame.K_a] or keys[pygame.K_LEFT]:
            self.x -= self.speed
        if keys[pygame.K_d] or keys[pygame.K_RIGHT]:
            self.x += self.speed
        self.x = max(boundary_x, min(boundary_x + boundary_size - self.size, self.x))
        self.y = max(boundary_y, min(boundary_y + boundary_size - self.size, self.y))

    def move_to_mouse(self, mouse_pos, boundary_x, boundary_y, boundary_size):
        mouse_x, mouse_y = mouse_pos
        if abs(self.x - mouse_x) > self.speed:
            self.x += self.speed if self.x < mouse_x else -self.speed
        if abs(self.y - mouse_y) > self.speed:
            self.y += self.speed if self.y < mouse_y else -self.speed
        self.x = max(boundary_x, min(boundary_x + boundary_size - self.size, self.x))
        self.y = max(boundary_y, min(boundary_y + boundary_size - self.size, self.y))

    def draw(self, surface):
        transparent_surface = pygame.Surface((self.size, self.size), pygame.SRCALPHA)
        pygame.draw.rect(transparent_surface, WHITE, (0, 0, self.size, self.size), 2)
        health_height = (self.health / self.full_health) * self.size
        pygame.draw.rect(transparent_surface, GREEN, (0, self.size - health_height, self.size, health_height))
        surface.blit(transparent_surface, (self.x, self.y))

class Enemy:
    def __init__(self, x, y, dx, dy, radius):
        self.x = x
        self.y = y
        self.dx = dx
        self.dy = dy
        self.radius = radius

    def update(self):
        self.x += self.dx
        self.y += self.dy

    def draw(self, surface):
        pygame.draw.circle(surface, RED, (int(self.x), int(self.y)), self.radius)

# ---------------------- Prediction ---------------------- #
def predict_positions(recorded_positions, start_x, start_y, speed, boundary_x, boundary_y, boundary_size):
    x, y = start_x, start_y
    predicted = []
    for mouse_x, mouse_y in recorded_positions:
        if abs(x - mouse_x) > speed:
            x += speed if x < mouse_x else -speed
        if abs(y - mouse_y) > speed:
            y += speed if y < mouse_y else -speed
        x = max(boundary_x, min(boundary_x + boundary_size - 50, x))
        y = max(boundary_y, min(boundary_y + boundary_size - 50, y))
        predicted.append((x, y))
    return predicted

# ---------------------- Game Class ---------------------- #
class Game:
    def __init__(self, mode="Play", outer_square_size=325, enemy_spawn_rate=5, enemy_speed=5, use_mouse_control=True, player_speed=5, full_health=5):
        self.mode = mode
        self.outer_square_size = outer_square_size
        self.outer_square_x = (WIDTH - self.outer_square_size) // 2
        self.outer_square_y = (HEIGHT - self.outer_square_size) // 2

        health = full_health * 15 if self.mode == "Record" else full_health
        self.player = Player(
            self.outer_square_x + (self.outer_square_size - 50) // 2,
            self.outer_square_y + (self.outer_square_size - 50) // 2,
            50,
            player_speed,
            health
        )

        self.enemies = []
        self.score = 0
        self.enemy_spawn_rate = enemy_spawn_rate
        self.enemy_speed = enemy_speed
        self.frame_count = 0
        self.running = True
        self.use_mouse_control = use_mouse_control

        self.recorded_positions = []
        self.predicted_positions = []  # for AI runtime shifting

        if self.mode == "AI":
            files = [f for f in os.listdir(RECORD_FOLDER) if f.endswith(".csv")]
            if files:
                chosen_file = random.choice(files)
                self.recorded_positions = load_recording(os.path.join(RECORD_FOLDER, chosen_file))
                # print(f"🎯 Using recording: {chosen_file}")

    def spawn_enemy(self):
        def circle_rect_collide(cx, cy, cr, rx, ry, rw, rh):
            closest_x = max(rx, min(cx, rx + rw))
            closest_y = max(ry, min(cy, ry + rh))
            distance = math.hypot(cx - closest_x, cy - closest_y)
            return distance < cr
    
        def will_collide(enemy_x, enemy_y, dx, dy, radius, player_future, player_size=50, safety_margin=10):
            safety_margin = 1
            ex, ey = float(enemy_x), float(enemy_y)
            frames_x = float('inf')
            frames_y = float('inf')
            if dx > 0:
                frames_x = (WIDTH + radius - enemy_x) / dx
            elif dx < 0:
                frames_x = (-radius - enemy_x) / dx
            if dy > 0:
                frames_y = (HEIGHT + radius - enemy_y) / dy
            elif dy < 0:
                frames_y = (-radius - enemy_y) / dy
            frames_to_check = int(min(frames_x, frames_y))
            for frame in range(frames_to_check):
                ex += dx
                ey += dy
                if ex < -radius * 2 or ex > WIDTH + radius * 2 or ey < -radius * 2 or ey > HEIGHT + radius * 2:
                    break
                if frame < len(player_future) - 1:
                    px1, py1 = player_future[frame]
                    px2, py2 = player_future[frame + 1]
                    t = (frame % 1.0)
                    px = px1 + (px2 - px1) * t
                    py = py1 + (py2 - py1) * t
                else:
                    px, py = player_future[-1]
                player_rect_x = px - safety_margin
                player_rect_y = py - safety_margin
                player_rect_w = player_size + 2 * safety_margin
                player_rect_h = player_size + 2 * safety_margin
                if circle_rect_collide(ex, ey, radius, player_rect_x, player_rect_y, player_rect_w, player_rect_h):
                    return True
            return False

        radius = 10
        attempts = 0
        MAX_ATTEMPTS = 40
        MIN_DIST = 60
        safe_enemy_found = False
    
        while attempts < MAX_ATTEMPTS:
            side = random.choice(["left", "right", "top", "bottom"])
            if side == "left":
                x = self.outer_square_x
                y = random.uniform(self.outer_square_y, self.outer_square_y + self.outer_square_size)
                angle = random.uniform(-math.pi/4, math.pi/4)
            elif side == "right":
                x = self.outer_square_x + self.outer_square_size
                y = random.uniform(self.outer_square_y, self.outer_square_y + self.outer_square_size)
                angle = random.uniform(3*math.pi/4, 5*math.pi/4)
            elif side == "top":
                x = random.uniform(self.outer_square_x, self.outer_square_x + self.outer_square_size)
                y = self.outer_square_y
                angle = random.uniform(math.pi/4, 3*math.pi/4)
            else:
                x = random.uniform(self.outer_square_x, self.outer_square_x + self.outer_square_size)
                y = self.outer_square_y + self.outer_square_size
                angle = random.uniform(-3*math.pi/4, -math.pi/4)
    
            dx = self.enemy_speed * math.cos(angle)
            dy = self.enemy_speed * math.sin(angle)
    
            if math.hypot(x - self.player.x, y - self.player.y) < MIN_DIST:
                attempts += 1
                continue
    
            if self.mode == "AI" and self.predicted_positions:
                if will_collide(x, y, dx, dy, radius, self.predicted_positions, player_size=self.player.size, safety_margin=10):
                    attempts += 1
                    continue
    
            self.enemies.append(Enemy(x, y, dx, dy, radius))
            safe_enemy_found = True
            break
    
        if not safe_enemy_found:
            pass
            # print("⚠️ No safe enemy spawn found this frame.")

    def run(self):
        try:
            while self.running:
                screen.fill(BLACK)
                for event in pygame.event.get():
                    if event.type == pygame.QUIT:
                        self.running = False
    
                if self.mode == "Play":
                    keys = pygame.key.get_pressed()
                    if self.use_mouse_control:
                        self.player.move_to_mouse(pygame.mouse.get_pos(), self.outer_square_x, self.outer_square_y, self.outer_square_size)
                    else:
                        self.player.move(keys, self.outer_square_x, self.outer_square_y, self.outer_square_size)
    
                elif self.mode == "Record":
                    self.player.move_to_mouse(pygame.mouse.get_pos(), self.outer_square_x, self.outer_square_y, self.outer_square_size)
                    self.recorded_positions.append(pygame.mouse.get_pos())
    
                elif self.mode == "AI":
                    if self.frame_count == 0 and self.recorded_positions:
                        self.recorded_positions = self.recorded_positions * 50
                        self.predicted_positions = predict_positions(
                        self.recorded_positions,
                        self.player.x, self.player.y,
                        self.player.speed,
                        self.outer_square_x, self.outer_square_y, self.outer_square_size
                        )
                
                    if self.predicted_positions:
                        next_pos = self.predicted_positions.pop(0)
                        self.player.move_to_mouse(next_pos, self.outer_square_x, self.outer_square_y, self.outer_square_size)
    
                self.frame_count += 1
                if self.frame_count % self.enemy_spawn_rate == 0:
                    self.spawn_enemy()

    
                for enemy in self.enemies[:]:
                    enemy.update()
                    if self.player.x < enemy.x < self.player.x + self.player.size and self.player.y < enemy.y < self.player.y + self.player.size:
                        collision_sound.play()
                        self.player.health -= 1
                        self.enemies.remove(enemy)
                    if enemy.x < -enemy.radius or enemy.x > WIDTH + enemy.radius or enemy.y < -enemy.radius or enemy.y > HEIGHT + enemy.radius:
                        self.enemies.remove(enemy)
    
                self.score += 1 / FPS
                pygame.draw.rect(screen, WHITE, (self.outer_square_x, self.outer_square_y, self.outer_square_size, self.outer_square_size), 2)
                self.player.draw(screen)
                for enemy in self.enemies:
                    enemy.draw(screen)
    
                font = pygame.font.Font(None, 36)
                screen.blit(font.render(f"Score: {int(self.score)}", True, WHITE), (10, 10))
                screen.blit(font.render(f"Mode: {self.mode}", True, WHITE), (10, 50))
    
                if self.player.health <= 0:
                    self.running = False
    
                pygame.display.flip()
                clock.tick(FPS)
    
            if self.mode == "Record" and self.recorded_positions:
                fname = os.path.join(RECORD_FOLDER, f"recording_{random.randint(10000, 99999)}.csv")
                record_inputs_to_a_file(self.recorded_positions, fname)
    
            print("Your final score is " + str(int(self.score)))
            
        except Exception as e:
            print("❌ An error occurred:", e)
            traceback.print_exc()
    
        finally:
            # Always clean up pygame, even after errors
            pygame.quit()
            sys.exit()


# ---------------------- Run ---------------------- #
if __name__ == "__main__":
    game_modes = ['Play', 'Record', 'AI']
    GAME_MODE = game_modes[2]
    ez_mode = True
    use_mouse_control = True
    full_health = 5
    outer_square_size = 300
    
    if ez_mode:
        enemy_spawn_rate = 5
        enemy_speed = 5
        player_speed = 5
    else:
        enemy_spawn_rate = 3
        enemy_speed = 15
        player_speed = 10
        
    game = Game(GAME_MODE, outer_square_size, enemy_spawn_rate, enemy_speed, use_mouse_control, player_speed, full_health)
    game.run()


Your final score is 4


SystemExit: 

In [77]:
# Why does it keep getting hit sometimes on hard mode
# Make the code cleaner